# 21 - Skill Extraction → PCA Bridge Validation

### Purpose
This notebook validates that **existing Chapter 0–1 artefacts** can be reused *unchanged* in a Chapter 3 context.  
It proves that user-style free text can be transformed into the **exact canonical skill representation** expected by downstream models.

This notebook performs **interface validation only**.  
It does **not** implement Chapter 3 logic or produce reusable code.

---

## Why this notebook exists
Chapter 3 introduces a new entry point: **user input**.  
Before building suitability, gaps, or competitiveness, we must ensure that:

user skill text
→ Chapter 0 skill extractor
→ 27 canonical skill flags (wide)
→ Chapter 1 PCA transformer
→ skill PCs


behaves **identically** to the pipeline used during model training.

This notebook removes the risk of silent misalignment.

---

## Scope (What is tested)

### Included
- Reuse of Chapter 0 skill extraction functions on user-style text
- Mapping to the 27 canonical skill groups
- Construction of a wide skill vector in the correct column order
- Projection into PCA space using the trained transformer
- Numerical equivalence with stored skill PCs for known job rows

### Explicitly excluded
- User schema definition
- Artefact loading framework
- Suitability scoring
- Competitiveness modelling
- Gap analysis
- Any `src/` implementation

---

## Steps Performed

### 1. Select a reference job
- Load a known job from the processed dataset
- Extract:
  - job description text
  - ground-truth 27 skill flags
  - stored skill PCs

This job acts as a **truth anchor**.

---

### 2. Run skill extraction on text
- Apply the existing Chapter 0 extraction pipeline
- Produce:
  - `skills_by_group` (27 P/A flags)
  - token-level matches for interpretability

---

### 3. Validate skill equivalence
- Compare extracted skill flags vs stored skill flags
- Reshape to long format for clarity
- Compute match rate

**Success criterion:** 100% match across all skill groups.

---

### 4. Project extracted skills into PCA space
- Convert extracted skills to wide format using canonical column order
- Apply the trained PCA transformer
- Obtain `skill_PC1 … skill_PC10`

---

### 5. Validate PCA equivalence
- Compare PCA outputs from extracted skills vs stored PCs
- Use tolerance-based comparison (rounding / numerical tolerance)

**Success criterion:** 100% match (within tolerance).

---

## Results & Conclusions

- Skill extraction behaves identically in Chapter 3 context
- Canonical skill naming and ordering are stable
- PCA transformer is compatible and reproducible
- The skill → PCA bridge is **reliable and complete**

No changes to existing code are required.

---

## Set Up

### Libraries

In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
#===
import sys
from pathlib import Path

### Path

In [2]:
project_root = Path().resolve().parent.parent
sys.path.append(str(project_root))
project_root

PosixPath('/Users/alejandrofp/Desktop/Projects/03_Flagship_Portfolio/job-intelligence-engine')

In [43]:
from src.job_intel.features.skill_extractor import extract_domain_level_flags
from src.job_intel.features.skill_extractor import explain_matches
from src.job_intel.features.skills_pca import transform_skills_to_pca
from src.job_intel.config import CH2_PROCESSED_DF

### Data

In [4]:

df = pd.read_csv(CH2_PROCESSED_DF)

In [5]:
for i in df.columns:
    print(i)

job_id
Job Description
Rating
Size
Founded
Industry
Sector
role_source
state
ownership_clean
job_description_clean
job_title_base
seniority_combined
job_title_norm
job_title_family
domain
sal_is_hourly
sal_min
sal_max
sal_mean
title_plus_description
core_programming__basic
core_programming__intermediate
core_programming__advanced
data_engineering_pipelines__basic
data_engineering_pipelines__intermediate
data_engineering_pipelines__advanced
ml_ai__basic
ml_ai__intermediate
ml_ai__advanced
analytics_stats__basic
analytics_stats__intermediate
analytics_stats__advanced
bi_viz__basic
bi_viz__intermediate
bi_viz__advanced
cloud__basic
cloud__intermediate
cloud__advanced
db_storage__basic
db_storage__intermediate
db_storage__advanced
productivity_workflow__basic
productivity_workflow__intermediate
productivity_workflow__advanced
soft_skills__core
soft_skills__leadership
domain_specific__none
Size_1 to 50 employees
Size_10000+ employees
Size_1001 to 5000 employees
Size_201 to 500 employees
Siz

## Stress test skill extraction

In [13]:
filtered_df = df[[  'job_id',
                    'Job Description',
                    'core_programming__basic',
                    'core_programming__intermediate',
                    'core_programming__advanced',
                    'data_engineering_pipelines__basic',
                    'data_engineering_pipelines__intermediate',
                    'data_engineering_pipelines__advanced',
                    'ml_ai__basic',
                    'ml_ai__intermediate',
                    'ml_ai__advanced',
                    'analytics_stats__basic',
                    'analytics_stats__intermediate',
                    'analytics_stats__advanced',
                    'bi_viz__basic',
                    'bi_viz__intermediate',
                    'bi_viz__advanced',
                    'cloud__basic',
                    'cloud__intermediate',
                    'cloud__advanced',
                    'db_storage__basic',
                    'db_storage__intermediate',
                    'db_storage__advanced',
                    'productivity_workflow__basic',
                    'productivity_workflow__intermediate',
                    'productivity_workflow__advanced',
                    'soft_skills__core',
                    'soft_skills__leadership',
                    'domain_specific__none']]
skill_cols = [      'core_programming__basic',
                    'core_programming__intermediate',
                    'core_programming__advanced',
                    'data_engineering_pipelines__basic',
                    'data_engineering_pipelines__intermediate',
                    'data_engineering_pipelines__advanced',
                    'ml_ai__basic',
                    'ml_ai__intermediate',
                    'ml_ai__advanced',
                    'analytics_stats__basic',
                    'analytics_stats__intermediate',
                    'analytics_stats__advanced',
                    'bi_viz__basic',
                    'bi_viz__intermediate',
                    'bi_viz__advanced',
                    'cloud__basic',
                    'cloud__intermediate',
                    'cloud__advanced',
                    'db_storage__basic',
                    'db_storage__intermediate',
                    'db_storage__advanced',
                    'productivity_workflow__basic',
                    'productivity_workflow__intermediate',
                    'productivity_workflow__advanced',
                    'soft_skills__core',
                    'soft_skills__leadership',
                    'domain_specific__none']

### Test

In [40]:
random_job_id = 123

description_test = (
    filtered_df
    .loc[filtered_df['job_id'] == random_job_id, 'Job Description']
    .iloc[0]
)

truth_long = (
    df.loc[[random_job_id], skill_cols]
      .melt(
          var_name="skill",
          value_name="truth_val"
      )
)

extracted = extract_domain_level_flags(description_test)

test_skills = pd.DataFrame({
    'skill': extracted.keys(),
    'pred_val': extracted.values()
})

eval_skill = truth_long.merge(test_skills, how='left', on='skill')

match_rate = (eval_skill['truth_val'] == eval_skill['pred_val']).mean() * 100
print(f"Match rate: {match_rate:.1f}%")


Match rate: 100.0%


### Extract fresh skills from text

In [41]:
explain_matches(description_test)

{'core_programming__basic': ['python', 'java', 'linux'],
 'core_programming__intermediate': ['scala'],
 'core_programming__advanced': [],
 'data_engineering_pipelines__basic': [],
 'data_engineering_pipelines__intermediate': ['big data', 'presto'],
 'data_engineering_pipelines__advanced': [],
 'ml_ai__basic': [],
 'ml_ai__intermediate': [],
 'ml_ai__advanced': [],
 'analytics_stats__basic': ['mathematics'],
 'analytics_stats__intermediate': ['pricing'],
 'analytics_stats__advanced': [],
 'bi_viz__basic': [],
 'bi_viz__intermediate': [],
 'bi_viz__advanced': [],
 'cloud__basic': ['aws', 'ec2'],
 'cloud__intermediate': [],
 'cloud__advanced': [],
 'db_storage__basic': ['sql'],
 'db_storage__intermediate': ['bigquery'],
 'db_storage__advanced': [],
 'productivity_workflow__basic': ['streamlined'],
 'productivity_workflow__intermediate': ['git'],
 'productivity_workflow__advanced': [],
 'soft_skills__core': ['coaching',
  'training',
  'communication',
  'presentation skills',
  'continuou

## Stress test PCA transformation

In [69]:
extracted = extract_domain_level_flags(description_test)

X_user = pd.DataFrame([{k: extracted.get(k, 0) for k in skill_cols}], columns=skill_cols)

user_pca = transform_skills_to_pca(X_user)

pca = df[[ 'job_id',
                'skill_PC1',
                'skill_PC2',
                'skill_PC3',
                'skill_PC4',
                'skill_PC5',
                'skill_PC6',
                'skill_PC7',
                'skill_PC8',
                'skill_PC9',
                'skill_PC10']]

pca_random_test = (
    pca
    .loc[filtered_df['job_id'] == random_job_id]
)

pca_cols = ['skill_PC1',
                'skill_PC2',
                'skill_PC3',
                'skill_PC4',
                'skill_PC5',
                'skill_PC6',
                'skill_PC7',
                'skill_PC8',
                'skill_PC9',
                'skill_PC10']

pca_truth = (
    pca_random_test[pca_cols]
      .melt(
          var_name="pca",
          value_name="truth_val"
      )
)

pca_user = (
    user_pca[pca_cols]
      .melt(
          var_name="pca",
          value_name="user_val"
      )
)

eval_pca = pca_truth.merge(pca_user, how='left', on='pca')

match_rate = (eval_pca['truth_val'].round(3) == eval_pca['user_val'].round(3)).mean() * 100
print(f"Match rate: {match_rate:.1f}%")



Match rate: 100.0%
